<a href="https://colab.research.google.com/github/Hkd225/Blood-Cell-Anomaly-Detection/blob/main/Blood_Cell_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import json
import shutil
import warnings
import joblib
import kagglehub
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    StratifiedKFold,
    GroupKFold,
    cross_val_score
)
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils import shuffle

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

warnings.filterwarnings("ignore")


In [2]:
def normalize_label(text):
    text = str(text).strip().lower().replace("_", " ")
    return " ".join(text.split())


def make_safe_filename(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

In [3]:
dataset_slug = "alitaqishah/blood-cell-anomaly-detection-2025"
dataset_name = dataset_slug.split("/")[-1]
output_dir = f"{dataset_name}_models"

path = kagglehub.dataset_download(dataset_slug)

main_file = os.path.join(path, "blood_cell_anomaly_detection.csv")
ref_file = os.path.join(path, "cell_type_reference.csv")

df = pd.read_csv(main_file)
df_ref = pd.read_csv(ref_file) if os.path.exists(ref_file) else None

print("Shape dataset:", df.shape)
print("Output folder :", output_dir)

100%|██████████| 380k/380k [00:00<00:00, 773kB/s]

Extracting files...
Shape dataset: (5880, 36)
Output folder : blood-cell-anomaly-detection-2025_models


In [4]:
possible_target_cols = ["cell_type", "celltype", "cell_class", "class", "label", "target"]

target_col = next((c for c in possible_target_cols if c in df.columns), None)
if target_col is None:
    raise ValueError(f"Kolom target tidak ditemukan. Kandidat: {possible_target_cols}")

print("Target col:", target_col)

Target col: cell_type


In [5]:
manual_mapping_raw = {
    "Neutrophil": "Normal",
    "Lymphocyte": "Normal",
    "Monocyte": "Normal",
    "Eosinophil": "Normal",
    "Basophil": "Normal",
    "Normal RBC": "Normal",
    "Platelet": "Normal",

    "Blast Cell": "Leukemia",
    "Prolymphocyte": "Leukemia",

    "Elliptocyte": "Anemia",
    "Schistocyte": "Anemia",
    "Spherocyte": "Anemia",
    "Target Cell": "Anemia",

    "Sickle Cell": "Sickle Cell Disease",

    "Hypersegmented Neutrophil": "Infection",
    "Toxic Granulation": "Infection",
    "Reactive Lymphocyte": "Infection",

    "Smudge Cell": "Artefact",
    "Artefact": "Artefact"
}
manual_mapping = {normalize_label(k): v for k, v in manual_mapping_raw.items()}

if df_ref is not None:
    ref_cols_lower = {c.lower(): c for c in df_ref.columns}

    ref_cell_col = next(
        (ref_cols_lower[c] for c in ["cell_type", "celltype", "cell class", "class", "label"] if c in ref_cols_lower),
        None
    )
    ref_disease_col = next(
        (ref_cols_lower[c] for c in ["disease_level", "disease_group", "disease", "group", "category"] if c in ref_cols_lower),
        None
    )

    if ref_cell_col is not None and ref_disease_col is not None:
        mapping_from_ref = {
            normalize_label(cell): disease
            for cell, disease in zip(df_ref[ref_cell_col], df_ref[ref_disease_col])
        }
        print("Mapping disease level diambil dari reference file.")
    else:
        mapping_from_ref = manual_mapping
        print("Reference file ada, tapi kolom mapping tidak cocok. Pakai mapping manual.")
else:
    mapping_from_ref = manual_mapping
    print("Reference file tidak ada. Pakai mapping manual.")

df["cell_type_normalized"] = df[target_col].apply(normalize_label)
df["disease_level"] = df["cell_type_normalized"].map(mapping_from_ref)

unmapped = df[df["disease_level"].isna()][[target_col, "cell_type_normalized"]].drop_duplicates()
if len(unmapped) > 0:
    print("\nCell type yang belum termapping:")
    print(unmapped)
    raise ValueError("Masih ada cell type yang belum termapping.")

print("\nDistribusi target:")
print(df["disease_level"].value_counts(normalize=True))

Reference file ada, tapi kolom mapping tidak cocok. Pakai mapping manual.

Distribusi target:
disease_level
Normal                 0.680272
Anemia                 0.110544
Leukemia               0.078231
Infection              0.076531
Artefact               0.030612
Sickle Cell Disease    0.023810
Name: proportion, dtype: float64


In [6]:
dup_count = df.duplicated().sum()
print("\nDuplikat penuh:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplikat penuh dihapus.")


Duplikat penuh: 0


In [7]:
possible_group_cols = [
    "patient_id", "patientid",
    "sample_id", "sampleid",
    "slide_id", "slideid",
    "record_id", "recordid",
    "specimen_id", "specimenid",
    "case_id", "caseid",
    "image_id", "imageid"
]

group_col = next((c for c in possible_group_cols if c in df.columns), None)
print("\nGroup col:", group_col)


Group col: None


In [8]:
explicit_drop_cols = {
    target_col,
    "cell_type_normalized",
    "disease_level",
    "disease_category",
    "anomaly_label",
    "cytodiffusion_anomaly_score",
    "cytodiffusion_classification_confidence"
}

possible_id_cols = [
    "id", "sample_id", "sampleid", "cell_id", "cellid",
    "record_id", "recordid", "patient_id", "patientid",
    "slide_id", "slideid", "image_id", "imageid",
    "case_id", "caseid", "specimen_id", "specimenid"
]
for col in possible_id_cols:
    if col in df.columns:
        explicit_drop_cols.add(col)

leak_keywords = [
    "target", "label", "class", "cell_type", "celltype",
    "disease", "diagnosis", "anomaly",
    "confidence", "prob", "probability", "score",
    "prediction", "predicted"
]

auto_drop_cols = set()
for col in df.columns:
    col_norm = col.strip().lower()
    if col in explicit_drop_cols:
        continue
    if any(k in col_norm for k in leak_keywords):
        auto_drop_cols.add(col)

shortcut_cols = {
    "dataset_source",
    "staining_protocol",
    "microscope_model",
    "magnification_x",
    "image_resolution_px"
}
for col in shortcut_cols:
    if col in df.columns:
        auto_drop_cols.add(col)

all_drop_cols = sorted(explicit_drop_cols.union(auto_drop_cols))

print("\nKolom yang di-drop:")
print(all_drop_cols)

X = df.drop(columns=[c for c in all_drop_cols if c in df.columns]).copy()
y = df["disease_level"].copy()

if X.shape[1] == 0:
    raise ValueError("Semua fitur habis ter-drop.")

print("\nFitur final:")
print(X.columns.tolist())


Kolom yang di-drop:
['anomaly_label', 'cell_id', 'cell_type', 'cell_type_normalized', 'cytodiffusion_anomaly_score', 'cytodiffusion_classification_confidence', 'dataset_source', 'disease_category', 'disease_level', 'granularity_score', 'image_resolution_px', 'labeller_confidence_score', 'lobularity_score', 'magnification_x', 'microscope_model', 'staining_protocol']

Fitur final:
['cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl']


In [9]:
if group_col is not None:
    groups = df[group_col].astype(str)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)
    groups_train = groups.iloc[train_idx].reset_index(drop=True)
    groups_test = groups.iloc[test_idx].reset_index(drop=True)

    overlap = set(groups_train).intersection(set(groups_test))
    print("\nOverlap group train-test:", len(overlap))
    if len(overlap) > 0:
        raise ValueError("Masih ada overlap group.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    groups_train = None
    groups_test = None

In [10]:
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

print("\nCategorical cols:", categorical_cols)
print("Numeric cols    :", numeric_cols)

target_encoder = LabelEncoder()
y_train_enc = target_encoder.fit_transform(y_train)
y_test_enc = target_encoder.transform(y_test)

print("\nMapping target:")
for idx, cls in enumerate(target_encoder.classes_):
    print(f"{idx} -> {cls}")


Categorical cols: ['patient_age_group', 'patient_sex']
Numeric cols    : ['cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl']

Mapping target:
0 -> Anemia
1 -> Artefact
2 -> Infection
3 -> Leukemia
4 -> Normal
5 -> Sickle Cell Disease


In [11]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
], remainder="drop")

In [12]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        random_state=42,
        class_weight="balanced"
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42,
        class_weight="balanced"
    )
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )
    print("\nXGBoost berhasil ditambahkan.")
except ImportError:
    print("\nXGBoost belum terpasang. Install dengan: pip install xgboost")


XGBoost berhasil ditambahkan.


In [13]:
if group_col is not None:
    n_groups = groups_train.nunique()
    cv_splits = min(5, n_groups)
    if cv_splits < 2:
        raise ValueError("Jumlah group train terlalu sedikit untuk CV.")
    cv = GroupKFold(n_splits=cv_splits)
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
trained_models = {}

for model_name, model in models.items():
    print(f"\nTraining + CV: {model_name}")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    if group_col is not None:
        scores = cross_val_score(
            pipe,
            X_train,
            y_train_enc,
            cv=cv,
            groups=groups_train,
            scoring="f1_macro"
        )
    else:
        scores = cross_val_score(
            pipe,
            X_train,
            y_train_enc,
            cv=cv,
            scoring="f1_macro"
        )

    cv_results[model_name] = {
        "mean_f1_macro": float(np.mean(scores)),
        "std_f1_macro": float(np.std(scores)),
        "scores": scores.tolist()
    }

    fitted = clone(pipe)
    fitted.fit(X_train, y_train_enc)
    trained_models[model_name] = fitted

print("\nCV results:")
print(json.dumps(cv_results, indent=2))

best_model_name = max(cv_results, key=lambda k: cv_results[k]["mean_f1_macro"])
best_model = trained_models[best_model_name]

print("\nBest model:", best_model_name)


Training + CV: Random Forest

Training + CV: Logistic Regression

Training + CV: SVM

Training + CV: XGBoost

CV results:
{
  "Random Forest": {
    "mean_f1_macro": 0.8854099847527582,
    "std_f1_macro": 0.013115769747699558,
    "scores": [
      0.8948395016006563,
      0.8974435277955931,
      0.8897371284475026,
      0.8607866337049165,
      0.8842431322151229
    ]
  },
  "Logistic Regression": {
    "mean_f1_macro": 0.7118187106431624,
    "std_f1_macro": 0.017378650559680044,
    "scores": [
      0.7196993749616283,
      0.7005087956004186,
      0.7109666773564172,
      0.6883646484025837,
      0.7395540568947644
    ]
  },
  "SVM": {
    "mean_f1_macro": 0.7498892064424723,
    "std_f1_macro": 0.013806490977890058,
    "scores": [
      0.7576337962679093,
      0.7457428445016343,
      0.7552142061187778,
      0.7253308595929414,
      0.7655243257310991
    ]
  },
  "XGBoost": {
    "mean_f1_macro": 0.910180430298072,
    "std_f1_macro": 0.009565523760371952,
  

In [14]:
y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test_enc, y_pred)
test_bal_acc = balanced_accuracy_score(y_test_enc, y_pred)
test_macro_f1 = f1_score(y_test_enc, y_pred, average="macro")
test_weighted_f1 = f1_score(y_test_enc, y_pred, average="weighted")

print("\n=== FINAL TEST METRICS ===")
print("Accuracy         :", test_acc)
print("Balanced Accuracy:", test_bal_acc)
print("Macro F1         :", test_macro_f1)
print("Weighted F1      :", test_weighted_f1)

print("\nClassification report:")
print(classification_report(y_test_enc, y_pred, target_names=target_encoder.classes_))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_enc, y_pred))


=== FINAL TEST METRICS ===
Accuracy         : 0.9379251700680272
Balanced Accuracy: 0.8716849816849818
Macro F1         : 0.9078152479528191
Weighted F1      : 0.9331169677989477

Classification report:
                     precision    recall  f1-score   support

             Anemia       0.95      0.86      0.90       130
           Artefact       1.00      0.92      0.96        36
          Infection       0.91      0.53      0.67        90
           Leukemia       0.99      1.00      0.99        92
             Normal       0.93      0.99      0.96       800
Sickle Cell Disease       1.00      0.93      0.96        28

           accuracy                           0.94      1176
          macro avg       0.96      0.87      0.91      1176
       weighted avg       0.94      0.94      0.93      1176


Confusion matrix:
[[112   0   0   0  18   0]
 [  1  33   0   1   1   0]
 [  0   0  48   0  42   0]
 [  0   0   0  92   0   0]
 [  3   0   5   0 792   0]
 [  2   0   0   0   0  26]]


In [15]:
sanity_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

y_train_shuffled = shuffle(y_train_enc, random_state=42)
sanity_model.fit(X_train, y_train_shuffled)
sanity_pred = sanity_model.predict(X_test)

sanity_acc = accuracy_score(y_test_enc, sanity_pred)
sanity_bal_acc = balanced_accuracy_score(y_test_enc, sanity_pred)
sanity_macro_f1 = f1_score(y_test_enc, sanity_pred, average="macro")
sanity_weighted_f1 = f1_score(y_test_enc, sanity_pred, average="weighted")

print("\n=== SANITY CHECK (SHUFFLED LABELS) ===")
print("Sanity Accuracy         :", sanity_acc)
print("Sanity Balanced Accuracy:", sanity_bal_acc)
print("Sanity Macro F1         :", sanity_macro_f1)
print("Sanity Weighted F1      :", sanity_weighted_f1)

sanity_pred_labels = target_encoder.inverse_transform(sanity_pred)
print("\nDistribusi prediksi sanity:")
print(pd.Series(sanity_pred_labels).value_counts(normalize=True))

if sanity_bal_acc > 0.30 or sanity_macro_f1 > 0.30:
    print("WARNING: Sanity metrics masih cukup tinggi. Audit shortcut/leakage/spurious correlation lebih lanjut.")
else:
    print("Sanity check sehat: tidak ada sinyal kuat leakage dari shuffle-label test.")


=== SANITY CHECK (SHUFFLED LABELS) ===
Sanity Accuracy         : 0.6802721088435374
Sanity Balanced Accuracy: 0.16666666666666666
Sanity Macro F1         : 0.1349527665317139
Sanity Weighted F1      : 0.5508276184967914

Distribusi prediksi sanity:
Normal    1.0
Name: proportion, dtype: float64
Sanity check sehat: tidak ada sinyal kuat leakage dari shuffle-label test.


In [16]:
os.makedirs(output_dir, exist_ok=True)

# simpan semua model terlatih
saved_model_files = {}
for model_name, model_pipeline in trained_models.items():
    safe_name = make_safe_filename(model_name)
    model_path = os.path.join(output_dir, f"model_{safe_name}.pkl")
    joblib.dump(model_pipeline, model_path)
    saved_model_files[model_name] = model_path

# simpan model terbaik juga dengan nama khusus
best_model_path = os.path.join(output_dir, "best_model.pkl")
joblib.dump(best_model, best_model_path)

# simpan artefak lain
joblib.dump(target_encoder, os.path.join(output_dir, "target_encoder.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(output_dir, "feature_columns.pkl"))
joblib.dump(categorical_cols, os.path.join(output_dir, "categorical_cols.pkl"))
joblib.dump(numeric_cols, os.path.join(output_dir, "numeric_cols.pkl"))
joblib.dump(group_col, os.path.join(output_dir, "group_col.pkl"))
joblib.dump(all_drop_cols, os.path.join(output_dir, "dropped_columns.pkl"))
joblib.dump(cv_results, os.path.join(output_dir, "cv_results.pkl"))

metadata = {
    "dataset_slug": dataset_slug,
    "dataset_name": dataset_name,
    "output_dir": output_dir,
    "best_model_name": best_model_name,
    "cv_metric": "f1_macro",
    "test_accuracy": float(test_acc),
    "test_balanced_accuracy": float(test_bal_acc),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "shuffle_label_accuracy": float(sanity_acc),
    "shuffle_label_balanced_accuracy": float(sanity_bal_acc),
    "shuffle_label_macro_f1": float(sanity_macro_f1),
    "shuffle_label_weighted_f1": float(sanity_weighted_f1),
    "group_col": group_col,
    "dropped_columns": all_drop_cols,
    "feature_columns": X.columns.tolist(),
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "saved_model_files": saved_model_files
}

with open(os.path.join(output_dir, "training_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# simpan report text
with open(os.path.join(output_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write("=== FINAL TEST METRICS ===\n")
    f.write(f"Accuracy         : {test_acc}\n")
    f.write(f"Balanced Accuracy: {test_bal_acc}\n")
    f.write(f"Macro F1         : {test_macro_f1}\n")
    f.write(f"Weighted F1      : {test_weighted_f1}\n\n")
    f.write("=== CLASSIFICATION REPORT ===\n")
    f.write(classification_report(y_test_enc, y_pred, target_names=target_encoder.classes_))
    f.write("\n\n=== CONFUSION MATRIX ===\n")
    f.write(np.array2string(confusion_matrix(y_test_enc, y_pred)))

print(f"\nSemua model dan metadata disimpan di folder: {output_dir}")



Semua model dan metadata disimpan di folder: blood-cell-anomaly-detection-2025_models


In [17]:
zip_base_name = output_dir
zip_file = shutil.make_archive(zip_base_name, "zip", output_dir)

print(f"ZIP berhasil dibuat: {zip_file}")

from google.colab import files
files.download(zip_file)

print("\nHa'ah lah.")

ZIP berhasil dibuat: /content/blood-cell-anomaly-detection-2025_models.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Ha'ah lah.
